<a href="https://www.kaggle.com/code/kiza123123/kaggle-notebooks-s6e4-grandmaster-submission-ipynb?scriptVersionId=314917037" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Playground Series S6E4 — Grandmaster Kaggle Notebook

Clean, reproducible pipeline for multi-class irrigation prediction.

What this notebook does:
- loads the competition data
- builds robust feature engineering
- trains LightGBM, XGBoost, and CatBoost with Stratified CV
- blends OOF probabilities
- tunes class thresholds on OOF predictions
- generates a final submission


In [40]:
import os
import gc
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

try:
    import lightgbm as lgb
except Exception:
    !pip -q install lightgbm
    import lightgbm as lgb

try:
    import xgboost as xgb
except Exception:
    !pip -q install xgboost
    import xgboost as xgb

try:
    from catboost import CatBoostClassifier
except Exception:
    !pip -q install catboost
    from catboost import CatBoostClassifier

USE_GPU = False
try:
    import torch
    USE_GPU = torch.cuda.is_available()
except Exception:
    USE_GPU = False

print('GPU available:', USE_GPU)

GPU available: True


In [41]:
DATA_DIRS = [
    '/kaggle/input/competitions/playground-series-s6e4',
    '/content',
    '.'
]

def find_file(filename: str):
    for d in DATA_DIRS:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{filename} not found in known data dirs')

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sample_path = find_file('sample_submission.csv')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

print('train:', train.shape)
print('test :', test.shape)
display(train.head(3))

train: (630000, 21)
test : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low


## Quick data check
We verify target balance, missing values, and basic schema.

In [42]:
target_col = 'Irrigation_Need'
id_col = 'id'

print('Missing values in train:', int(train.isna().sum().sum()))
print('Missing values in test :', int(test.isna().sum().sum()))
print('Duplicate ids in train  :', int(train[id_col].duplicated().sum()))
print('Duplicate ids in test   :', int(test[id_col].duplicated().sum()))

display(train[target_col].value_counts(normalize=True).rename('train_share').to_frame())

Missing values in train: 0
Missing values in test : 0
Duplicate ids in train  : 0
Duplicate ids in test   : 0


,train_share
Irrigation_Need,
Low,0.587170
Medium,0.379483
High,0.033348


In [43]:
LABELS = ['Low', 'Medium', 'High']
label_to_int = {k: i for i, k in enumerate(LABELS)}
int_to_label = {i: k for k, i in label_to_int.items()}
y = train[target_col].map(label_to_int).astype(int)

NUM_COLS = ['Soil_Moisture', 'Rainfall_mm', 'Temperature_C', 'Wind_Speed_kmh', 'Humidity']
CAT_COLS = [
    c for c in train.columns
    if c not in [id_col, target_col] and c not in NUM_COLS
]

print('Numeric cols:', NUM_COLS)
print('Categorical cols:', CAT_COLS)

Numeric cols: ['Soil_Moisture', 'Rainfall_mm', 'Temperature_C', 'Wind_Speed_kmh', 'Humidity']
Categorical cols: ['Soil_Type', 'Soil_pH', 'Organic_Carbon', 'Electrical_Conductivity', 'Sunlight_Hours', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region']


In [44]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['soil_lt_25'] = (df['Soil_Moisture'] < 25).astype(int)
    df['rain_lt_300'] = (df['Rainfall_mm'] < 300).astype(int)
    df['temp_gt_30'] = (df['Temperature_C'] > 30).astype(int)
    df['wind_gt_10'] = (df['Wind_Speed_kmh'] > 10).astype(int)

    df['high_score'] = 2 * df['soil_lt_25'] + 2 * df['rain_lt_300'] + df['temp_gt_30'] + df['wind_gt_10']
    df['is_harvest'] = (df['Crop_Growth_Stage'] == 'Harvest').astype(int)
    df['is_sowing'] = (df['Crop_Growth_Stage'] == 'Sowing').astype(int)
    df['mulch_yes'] = (df['Mulching_Used'] == 'Yes').astype(int)
    df['low_score'] = 2 * df['is_harvest'] + 2 * df['is_sowing'] + df['mulch_yes']
    df['magic_score'] = df['high_score'] - df['low_score']
    df['formula_pred_int'] = np.where(df['magic_score'] <= 0, 0, np.where(df['magic_score'] <= 3, 1, 2))

    df['heat_index'] = df['Temperature_C'] * (df['Humidity'] / 100.0)
    df['moisture_stress'] = df['Soil_Moisture'] / (df['Temperature_C'] + 1e-6)
    df['rain_per_temp'] = df['Rainfall_mm'] / (df['Temperature_C'] + 1e-6)
    df['dry_and_hot'] = ((df['Soil_Moisture'] < 20) & (df['Temperature_C'] > 35)).astype(int)
    df['wet_and_cool'] = ((df['Soil_Moisture'] > 60) & (df['Temperature_C'] < 20)).astype(int)

    for col in ['Soil_Type', 'Region']:
        df[f'{col}_mean_moisture'] = df.groupby(col)['Soil_Moisture'].transform('mean')
        df[f'{col}_moisture_diff'] = df['Soil_Moisture'] - df[f'{col}_mean_moisture']

    return df

train_fe = add_features(train)
test_fe = add_features(test)

FEATURES = [c for c in train_fe.columns if c not in [id_col, target_col]]
for c in CAT_COLS:
    train_fe[c] = train_fe[c].astype('category')
    test_fe[c] = test_fe[c].astype('category')

X = train_fe[FEATURES]
X_test = test_fe[FEATURES]

## Utility functions
We train with OOF probabilities so threshold tuning remains valid.

In [45]:
def oof_score(probs: np.ndarray, y_true: np.ndarray) -> float:
    return balanced_accuracy_score(y_true, probs.argmax(axis=1))

def tune_thresholds(y_true: np.ndarray, y_probs: np.ndarray):
    def objective(thresholds):
        weighted = y_probs * thresholds
        return -balanced_accuracy_score(y_true, weighted.argmax(axis=1))

    res = minimize(objective, x0=np.ones(y_probs.shape[1]), method='Nelder-Mead', tol=1e-6)
    return res.x

def clean_predict_proba(model, X_part):
    if hasattr(model, 'best_iteration_') and model.best_iteration_ is not None:
        try:
            return model.predict_proba(X_part, num_iteration=model.best_iteration_)
        except Exception:
            pass
    return model.predict_proba(X_part)

In [46]:
def cv_lgbm(X, y, X_test, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X), 3), dtype=np.float32)
    tst = np.zeros((len(X_test), 3), dtype=np.float32)
    scores = []

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        params = dict(
            objective='multiclass',
            num_class=3,
            n_estimators=12000,
            learning_rate=0.02,
            num_leaves=128,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            random_state=SEED + fold,
            n_jobs=-1,
            verbose=-1,
        )
        model = lgb.LGBMClassifier(**params)
        sw = compute_sample_weight(class_weight='balanced', y=y[tr])
        model.fit(
            X.iloc[tr], y[tr],
            sample_weight=sw,
            eval_set=[(X.iloc[va], y[va])],
            eval_metric='multi_logloss',
            callbacks=[lgb.early_stopping(250, verbose=False)]
        )
        pv = clean_predict_proba(model, X.iloc[va])
        pt = clean_predict_proba(model, X_test)
        oof[va] = pv
        tst += pt / n_splits
        sc = balanced_accuracy_score(y[va], pv.argmax(axis=1))
        scores.append(sc)
        print(f'LGBM fold {fold}: {sc:.6f}')
        del model, sw, pv, pt
        gc.collect()

    return oof, tst, scores

def cv_xgb(X, y, X_test, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X), 3), dtype=np.float32)
    tst = np.zeros((len(X_test), 3), dtype=np.float32)
    scores = []

    base_params = dict(
        objective='multi:softprob',
        num_class=3,
        n_estimators=15000,
        learning_rate=0.015,
        max_depth=8,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.5,
        tree_method='hist',
        eval_metric='mlogloss',
        random_state=SEED,
        n_jobs=-1,
    )
    if USE_GPU:
        base_params['device'] = 'cuda'

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        params = dict(base_params)
        params['random_state'] = SEED + fold
        model = xgb.XGBClassifier(**params)
        sw = compute_sample_weight(class_weight='balanced', y=y[tr])
        try:
            model.fit(
                X.iloc[tr], y[tr],
                sample_weight=sw,
                eval_set=[(X.iloc[va], y[va])],
                verbose=False,
            )
        except Exception:
            params.pop('device', None)
            model = xgb.XGBClassifier(**params)
            model.fit(
                X.iloc[tr], y[tr],
                sample_weight=sw,
                eval_set=[(X.iloc[va], y[va])],
                verbose=False,
            )

        pv = model.predict_proba(X.iloc[va])
        pt = model.predict_proba(X_test)
        oof[va] = pv
        tst += pt / n_splits
        sc = balanced_accuracy_score(y[va], pv.argmax(axis=1))
        scores.append(sc)
        print(f'XGB  fold {fold}: {sc:.6f}')
        del model, sw, pv, pt
        gc.collect()

    return oof, tst, scores

def cv_cat(X, y, X_test, cat_idx, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X), 3), dtype=np.float32)
    tst = np.zeros((len(X_test), 3), dtype=np.float32)
    scores = []
    task_type = 'GPU' if USE_GPU else 'CPU'

    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        model = CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric='MultiClass',
            iterations=15000,
            learning_rate=0.02,
            depth=8,
            random_seed=SEED + fold,
            task_type=task_type,
            auto_class_weights='Balanced',
            od_type='Iter',
            od_wait=300,
            verbose=0,
        )
        try:
            model.fit(
                X.iloc[tr], y[tr],
                cat_features=cat_idx if len(cat_idx) else None,
                eval_set=(X.iloc[va], y[va]),
                use_best_model=True,
                verbose=0,
            )
        except Exception:
            if task_type == 'GPU':
                model = CatBoostClassifier(
                    loss_function='MultiClass',
                    eval_metric='MultiClass',
                    iterations=15000,
                    learning_rate=0.02,
                    depth=8,
                    random_seed=SEED + fold,
                    task_type='CPU',
                    auto_class_weights='Balanced',
                    od_type='Iter',
                    od_wait=300,
                    verbose=0,
                )
                model.fit(
                    X.iloc[tr], y[tr],
                    cat_features=cat_idx if len(cat_idx) else None,
                    eval_set=(X.iloc[va], y[va]),
                    use_best_model=True,
                    verbose=0,
                )
            else:
                raise

        pv = model.predict_proba(X.iloc[va])
        pt = model.predict_proba(X_test)
        oof[va] = pv
        tst += pt / n_splits
        sc = balanced_accuracy_score(y[va], pv.argmax(axis=1))
        scores.append(sc)
        print(f'CAT  fold {fold}: {sc:.6f}')
        del model, pv, pt
        gc.collect()

    return oof, tst, scores

In [47]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

def cv_xgb(X, y, X_test, n_splits=5, random_state=42):

    oof = np.zeros(len(X))
    preds = np.zeros(len(X_test))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # --- FORCE SAFE COPY + DTYPE HANDLING ---
    X = X.copy()
    X_test = X_test.copy()

    # XGBoost safe categorical handling
    cat_cols = X.select_dtypes(include=["category"]).columns.tolist()

    for c in cat_cols:
        X[c] = X[c].astype("category")
        X_test[c] = X_test[c].astype("category")

    for fold, (tr, va) in enumerate(kf.split(X, y)):
        print(f"\nFold {fold + 1}")

        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y[tr], y[va]

        model = xgb.XGBClassifier(
            n_estimators=10_000,
            learning_rate=0.03,
            max_depth=7,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            tree_method="hist",     
            enable_categorical=True,
            eval_metric="auc",
            random_state=42,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=200,
            early_stopping_rounds=200
        )

        oof[va] = model.predict_proba(X_va)[:, 1]
        preds += model.predict_proba(X_test)[:, 1] / n_splits

        fold_score = roc_auc_score(y_va, oof[va])
        print("Fold AUC:", fold_score)

    final_score = roc_auc_score(y, oof)
    print("\nOOF AUC:", final_score)

    return oof, preds, final_score

## Train base models
We keep OOF probabilities for a stable ensemble and threshold tuning.

In [48]:
cat_idx = [X.columns.get_loc(c) for c in CAT_COLS if c in X.columns]


oof_xgb, tst_xgb, sc_xgb = cv_xgb(X, y.values, X_test, n_splits=5)
oof_lgb, tst_lgb, sc_lgb = cv_lgbm(X, y.values, X_test, n_splits=5)
oof_cat, tst_cat, sc_cat = cv_cat(X, y.values, X_test, cat_idx=cat_idx, n_splits=5)


print('XGB  OOF:', oof_score(oof_xgb, y.values))

print('LGBM OOF:', oof_score(oof_lgb, y.values))

print('CAT  OOF:', oof_score(oof_cat, y.values))


Fold 1


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:
weights = np.array([1.0 / oof_score(oof_lgb, y.values), 1.0 / oof_score(oof_xgb, y.values), 1.0 / oof_score(oof_cat, y.values)])
weights = weights / weights.sum()
print('Blend weights:', weights)

oof_blend = weights[0] * oof_lgb + weights[1] * oof_xgb + weights[2] * oof_cat
tst_blend = weights[0] * tst_lgb + weights[1] * tst_xgb + weights[2] * tst_cat

print('Blend OOF before thresholds:', oof_score(oof_blend, y.values))

In [ ]:
thresholds = tune_thresholds(y.values, oof_blend)
print('Optimized thresholds:', thresholds)

final_oof_preds = (oof_blend * thresholds).argmax(axis=1)
final_test_preds = (tst_blend * thresholds).argmax(axis=1)

print('Blend OOF after thresholds:', balanced_accuracy_score(y.values, final_oof_preds))

formula_test_preds = test_fe['formula_pred_int'].values
extreme_mask = (test_fe['magic_score'].values <= -2) | (test_fe['magic_score'].values >= 5)
final_test_preds = np.where(extreme_mask, formula_test_preds, final_test_preds)

submission = pd.DataFrame({
    'id': test[id_col].values,
    target_col: [int_to_label[i] for i in final_test_preds]
})
submission.to_csv('submission.csv', index=False)
display(submission.head())
print('Saved submission.csv')
print(submission[target_col].value_counts())

## Optional: compare submission distribution with train
This helps sanity-check calibration.

In [ ]:
train_dist = train[target_col].value_counts(normalize=True).rename('train_share')
sub_dist = submission[target_col].value_counts(normalize=True).rename('submission_share')
dist_compare = pd.concat([train_dist, sub_dist], axis=1).fillna(0)
dist_compare['diff'] = dist_compare['submission_share'] - dist_compare['train_share']
display(dist_compare)